# AbstractTensor — Manifolds and Tensors

This notebook tests and explains how to use objects defined in `manifolds.jl`, `indices.jl` and `tensors.jl` step by step.

---
## 1. Loading package

In [1]:
using AbstractTensors

[ Info: Precompiling AbstractTensors [55c8b40c-cbfa-4141-803e-4831c9841971] (cache misses: include_dependency fsize change (2), wrong source (6), mismatched flags (12))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


---
## 2. Manifolds and Vbundles 
---

### a. Manifold type and definition

#### Documentation access
- `@doc Manifold` — shows the docstring directly (julia macro)
- `?Manifold` — same, with search results prepended (Jupyter help mode)

In [2]:
@doc Manifold

```julia
Manifold
```

Struct representing a registered differentiable manifold. Instances are created by [`@def_manifold`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
M.name              # :M
M.dim               # 4
M.tangent_bundle    # :tangentM
M.cotangent_bundle  # :cotangentM
M.vbundles          # [:tangentM, :cotangentM]
```

### Fields

  * `name`             : manifold name, e.g. `:M`
  * `dim`              : dimension
  * `tangent_bundle`   : name of the tangent bundle, e.g. `:tangentM`
  * `cotangent_bundle` : name of the cotangent (dual) bundle, e.g. `:cotangentM`
  * `vbundles`         : all associated bundle names


In [3]:
?@def_manifold

```julia
@def_manifold name dim [idx1, idx2, ...]
```

Define a new manifold and automatically create its tangent and cotangent bundles. Bind the following variables in the caller's scope:

  * `name`            → a [`Manifold`](@ref) instance
  * `tangent<name>`   → a [`VBundle`](@ref) instance (`isdual = false`)
  * `cotangent<name>` → a [`VBundle`](@ref) instance (`isdual = true`)

Each index symbol is bound in the caller's scope as a contravariant [`TensorIndex`](@ref), so unary `-` can be used directly:

```julia
a1          # TensorIndex(:a1, :tangentM)   — contravariant
-a1         # TensorIndex(:a1, :cotangentM) — covariant
[a1, -a2]   # Vector{TensorIndex}           — uniform type
```

`dim` can be a concrete integer or a symbolic name for parametric manifolds.

Register `name` in `_MANIFOLDS`, the tangent and cotangent bundles in `_VBUNDLES`, and all index symbols in `_INDICES`.

#### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4]   # concrete dimension
@def_manifold M d [b1, b2, b3, b4]   # parametric dimension
```


In [4]:
@def_manifold M 2 [a1, a2, a3, a4]

Defined manifold M of dimension 2 with tangent bundle tangentM and cotangent bundle cotangentM


In [5]:
@show M.name
@show M.dim
@show M.tangent_bundle
@show M.cotangent_bundle
@show M.vbundles;

M.name = :M
M.dim = 2
M.tangent_bundle = :tangentM
M.cotangent_bundle = :cotangentM
M.vbundles = [:tangentM, :cotangentM]


In [6]:
@def_manifold N d [b1, b2, b3, b4]

Defined manifold N of dimension d with tangent bundle tangentN and cotangent bundle cotangentN


In [7]:
# Registries
display(_MANIFOLDS)
display(_VBUNDLES)
display(_INDICES)

Dict{Symbol, Manifold} with 2 entries:
  :N => Manifold(N, dim=d, TBundle=tangentN, CBundle=cotangentN)
  :M => Manifold(M, dim=2, TBundle=tangentM, CBundle=cotangentM)

Dict{Symbol, VBundle} with 4 entries:
  :cotangentM => VBundle(cotangentM, cotangent, dual=tangentM, manifold=M, dim=…
  :tangentM   => VBundle(tangentM, tangent, dual=cotangentM, manifold=M, dim=2)
  :cotangentN => VBundle(cotangentN, cotangent, dual=tangentN, manifold=N, dim=…
  :tangentN   => VBundle(tangentN, tangent, dual=cotangentN, manifold=N, dim=d)

Dict{Symbol, Symbol} with 8 entries:
  :a2 => :tangentM
  :b4 => :tangentN
  :a3 => :tangentM
  :b2 => :tangentN
  :a4 => :tangentM
  :b1 => :tangentN
  :a1 => :tangentM
  :b3 => :tangentN

In [8]:
M

Dimension,2
Tangent Bundle,tangentM
Cotangent Bundle,cotangentM
All VBundles,"tangentM, cotangentM"


In [9]:
@undef_manifold M

┌ Warning: Manifold M has been undefined. The variable M still holds a stale reference, accessing it will error. Restart the kernel to fully clear the binding.
└ @ Main ~/JuliaTensor/AbstractTensors/src/manifolds.jl:367


Undefined tangent bundle:   tangentM
Undefined cotangent bundle: cotangentM


In [10]:
M.name
M.dim
M.tangent_bundle
M.cotangent_bundle
M.vbundles

┌ Warning: Manifold :M has been undefined. Variable still holds a stale reference.
└ @ AbstractTensors ~/JuliaTensor/AbstractTensors/src/manifolds.jl:393
┌ Warning: Manifold :M has been undefined. Variable still holds a stale reference.
└ @ AbstractTensors ~/JuliaTensor/AbstractTensors/src/manifolds.jl:393
┌ Warning: Manifold :M has been undefined. Variable still holds a stale reference.
└ @ AbstractTensors ~/JuliaTensor/AbstractTensors/src/manifolds.jl:393
┌ Warning: Manifold :M has been undefined. Variable still holds a stale reference.
└ @ AbstractTensors ~/JuliaTensor/AbstractTensors/src/manifolds.jl:393
┌ Warning: Manifold :M has been undefined. Variable still holds a stale reference.
└ @ AbstractTensors ~/JuliaTensor/AbstractTensors/src/manifolds.jl:393


In [11]:
@def_manifold M 4 [a1, a2, a3, a4]

Defined manifold M of dimension 4 with tangent bundle tangentM and cotangent bundle cotangentM


In [12]:
@show M.name
@show M.dim
@show M.tangent_bundle
@show M.cotangent_bundle
@show M.vbundles;

M.name = :M
M.dim = 4
M.tangent_bundle = :tangentM
M.cotangent_bundle = :cotangentM
M.vbundles = [:tangentM, :cotangentM]


### b. Tangent and Cotangent bundles

In [13]:
@doc VBundle

```julia
VBundle
```

Struct representing a registered vector bundle. Instances are created by [`@def_manifold`](@ref) for the tangent and cotangent bundles, and bound to variables in the caller's scope (e.g. `tangentM`, `cotangentM`).

Provides dot access to all metadata:

```julia
tangentM.name      # :tangentM
tangentM.manifold  # :M
tangentM.dim       # 4
tangentM.isdual    # false
tangentM.dual      # :cotangentM
tangentM.indices   # [TensorIndex(:a1, :tangentM), ...]
```

### Fields

  * `name`     : bundle name, e.g. `:tangentM`
  * `manifold` : base manifold name, e.g. `:M`
  * `dim`      : fibre dimension
  * `isdual`   : false = contravariant (upper) slots, true = covariant (lower) slots.              Authoritative for index variance via [`is_up`](@ref) / [`is_down`](@ref).
  * `dual`     : name of the paired dual bundle, e.g. `:cotangentM` or `:dualE`
  * `indices`  : the `TensorIndex` objects living in this bundle


In [14]:
tangentM

VBundle(tangentM, tangent, dual=cotangentM, manifold=M, dim=4)

In [15]:
display(tangentM.name)
display(tangentM.manifold)
display(tangentM.dim)
display(tangentM.isdual)
display(tangentM.indices)

:tangentM

:M

4

false

4-element Vector{TensorIndex}:
  a1 ∈ tangentM
  a2 ∈ tangentM
  a3 ∈ tangentM
  a4 ∈ tangentM

In [16]:
cotangentM

VBundle(cotangentM, cotangent, dual=tangentM, manifold=M, dim=4)

In [17]:
display(cotangentM.name)
display(cotangentM.manifold)
display(cotangentM.dim)
display(cotangentM.isdual)
display(cotangentM.indices)

:cotangentM

:M

4

true

4-element Vector{TensorIndex}:
 -a1 ∈ cotangentM
 -a2 ∈ cotangentM
 -a3 ∈ cotangentM
 -a4 ∈ cotangentM

In [18]:
cotangentM.indices

4-element Vector{TensorIndex}:
 -a1 ∈ cotangentM
 -a2 ∈ cotangentM
 -a3 ∈ cotangentM
 -a4 ∈ cotangentM

### c. Definition of extra Vbundles

In [19]:
?@def_vbundle

```julia
@def_vbundle name manifold dim [idx1, idx2, ...]
```

Define a new vector bundle `name` of fibre dimension `dim` over `manifold`, and its dual bundle `dual<name>`. Bind the following variables in the caller's scope:

  * `name`       → a [`VBundle`](@ref) instance (`isdual = false`)
  * `dualname`   → a [`VBundle`](@ref) instance (`isdual = true`)

Each index symbol in `[idx1, idx2, ...]` is bound to a contravariant [`TensorIndex`](@ref) and registered to `name` (the primal bundle). The dual bundle shares the same index symbols — `-A1` resolves to `TensorIndex(:A1, :dualname)` via [`flip`](@ref) and the `dual` field on [`VBundle`](@ref).

`dim` accepts a concrete integer or a bare symbol for parametric bundles. The fibre dimension is independent of the base manifold dimension.

Registers both bundles in `_VBUNDLES` and appends their names to `manifold.vbundles`.

### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4]
@def_vbundle E M 3 [A1, A2, A3]    # rank-3 bundle and its dual over M
@def_vbundle E M r [A1, A2, A3]    # parametric fibre dimension

E.isdual       # false
dualE.isdual   # true
A1.vbundle     # :E
-A1            # TensorIndex(:A1, :dualE)
M.vbundles     # [:tangentM, :cotangentM, :E, :dualE]
```


In [20]:
@def_vbundle E M 3 [A1, A2, A3]

Defined VBundle E (dim=3) and dual dualE over manifold M


In [21]:
E

VBundle(E, tangent, dual=dualE, manifold=M, dim=3)

In [22]:
dualE

VBundle(dualE, cotangent, dual=E, manifold=M, dim=3)

## 3. Indices and tensors

### a. Indices

In [23]:
@doc TensorIndex

```julia
TensorIndex
```

An index symbol associated to a specific vector bundle, fully encoding its variance through the bundle it lives in:

  * vbundle = `:tangentM`   → contravariant (upper) index
  * vbundle = `:cotangentM` → covariant (lower) index

After `@def_manifold M 4 [a1, a2, a3, a4]`, each index symbol is bound in the caller's scope as a contravariant `TensorIndex`:

```julia
 a1          # TensorIndex(:a1, :tangentM)   — contravariant
-a1          # TensorIndex(:a1, :cotangentM) — covariant (unary -)
+a1          # TensorIndex(:a1, :tangentM)   — contravariant (unary +, identity)
flip(a1)     # Change vbundle of index to its dual
```

Bracket indexing uses bound `TensorIndex` values only: `F[-a1, -a2]`.

Direct construction when both fields are known:

```julia
TensorIndex(:a1, :tangentM)
```

### Fields

  * `symbol`  : the index name, e.g. `:a1`
  * `vbundle` : the bundle it lives in, e.g. `:tangentM` or `:cotangentM`

Dot access (via `getproperty`) also exposes:

```julia
idx.is_up    # upper / contravariant
idx.is_down  # lower / covariant
```


In [24]:
TensorIndex(:a1,:tangentM)

 a1 ∈ tangentM

In [25]:
a1

 a1 ∈ tangentM

In [26]:
+a1

 a1 ∈ tangentM

In [27]:
[+a1,-a2]

2-element Vector{TensorIndex}:
  a1 ∈ tangentM
 -a2 ∈ cotangentM

In [28]:
[a1,-a2]

2-element Vector{TensorIndex}:
  a1 ∈ tangentM
 -a2 ∈ cotangentM

In [29]:
flip(a1)

-a1 ∈ cotangentM

In [30]:
A1

 A1 ∈ E

In [31]:
[A1,A2,-A3]

3-element Vector{TensorIndex}:
  A1 ∈ E
  A2 ∈ E
 -A3 ∈ dualE

In [32]:
[-A1,+a2,-a3]

3-element Vector{TensorIndex}:
 -A1 ∈ dualE
  a2 ∈ tangentM
 -a3 ∈ cotangentM

### b. Tensors

In [34]:
?Tensor

search: Tensor is_tensor tensor_of TensorIndex entr gensym error nor Vector



```julia
Tensor
```

A registered abstract tensor. Instances are created by [`@def_tensor`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
T.manifold       # :M  (Symbol key into _MANIFOLDS)
T.slots          # [:cotangentM, :cotangentM]  — vbundle per slot
T.symmetries     # [SlotSymmetry(n=2, order=2)]
T.is_traceless   # false
T.known_traces   # Any[]  (populated later)
T.print_as       # :T
T.metric         # :g or nothing
T.rank           # 2      (derived — length of slots)
T.manifold_data  # the Manifold instance (derived — looks up _MANIFOLDS)
```

## Fields

  * `manifold`      : name of the base manifold, key into `_MANIFOLDS`
  * `slots`         : vbundle symbol per slot, encoding variance.                   `:cotangentM` → covariant, `:tangentM` → contravariant.                   Built from the index signs in the `@def_tensor` expression.
  * `symmetries`    : `Vector{`[`SlotSymmetry`](@ref)`}` — list of permutation                   groups acting on slot positions `1:rank`. Use convenience                   constructors `symmetric`, `antisymmetric`, etc.
  * `is_traceless`  : if `true`, any self-contraction of this tensor gives zero                   (e.g. the Weyl tensor).
  * `known_traces`  : user-declared trace values, e.g. `g[a,-a] = dim`.                   Format TBD — stored as `Any[]` until contraction is                   implemented.
  * `print_as`      : display name. Controls how the tensor appears in `show`                   and (later) LaTeX output.
  * `metric`        : name of the metric tensor associated with this tensor,                   a key into `_METRICS`, or `nothing` if no metric                   has been assigned. Required for raising/lowering indices.
